In [18]:
import pandas as pd

master = pd.read_csv(
    PROCESSED_DATA / "master_enso_monthly.csv",
    parse_dates=["Date"]
)

master.head()
print(features.columns.tolist())

['Date', 'nino34', 'nino3', 'nino4', 'iod', 'soi']


In [17]:
# FEATURE ENGINEERING
features = master.copy()

# Keep only rows where every climate index exists
features = features.dropna(
    subset=["nino34", "nino3", "nino4", "iod", "soi"]
).reset_index(drop=True)
features.info()
features.head()
features.tail()
features.shape


<class 'pandas.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    892 non-null    datetime64[us]
 1   nino34  892 non-null    float64       
 2   nino3   892 non-null    float64       
 3   nino4   892 non-null    float64       
 4   iod     892 non-null    float64       
 5   soi     892 non-null    float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 41.9 KB


(892, 6)

In [8]:
# CREATE LAG FEATURES
climate_columns = ["nino34", "nino3", "nino4", "iod", "soi"]

for col in climate_columns:
    for lag in [1, 2, 3]:
        features[f"{col}_lag{lag}"] = features[col].shift(lag)

In [9]:
# CREATE ROLLING FEATURES
for col in climate_columns:
    features[f"{col}_roll3"] = (
        features[col]
        .rolling(window=3)
        .mean()
    )

In [12]:
# CREATE TARGET
features["target"] = features["nino34"].shift(-1)
features.rename(
    columns={"target": "future_nino34"},
    inplace=True
)

In [13]:
#remove incomplete rows/missing values
features = features.dropna().reset_index(drop=True)

print(features.shape)
features.head()

(887, 27)


,Date,nino34,nino3,nino4,iod,soi,nino34_lag1,nino34_lag2,nino34_lag3,nino3_lag1,...,iod_lag3,soi_lag1,soi_lag2,soi_lag3,nino34_roll3,nino3_roll3,nino4_roll3,iod_roll3,soi_roll3,future_nino34
0,1951-04-01,-0.23,-0.21,-0.42,-0.513,-0.3,-0.38,-1.04,-1.30,-0.33,...,0.256,-0.1,0.9,1.5,-0.550000,-0.433333,-0.703333,-0.014333,0.166667,-0.01
1,1951-05-01,-0.01,-0.18,0.26,-0.138,-0.7,-0.23,-0.38,-1.04,-0.21,...,0.211,-0.3,-0.1,0.9,-0.206667,-0.240000,-0.246667,-0.130667,-0.366667,0.00
2,1951-06-01,0.00,0.04,0.08,-0.190,0.2,-0.01,-0.23,-0.38,-0.18,...,0.259,-0.7,-0.3,-0.1,-0.080000,-0.116667,-0.026667,-0.280333,-0.266667,0.30
3,1951-07-01,0.30,0.62,0.23,-0.220,-1.0,0.00,-0.01,-0.23,0.04,...,-0.513,0.2,-0.7,-0.3,0.096667,0.160000,0.190000,-0.182667,-0.500000,0.17
4,1951-08-01,0.17,0.41,-0.26,0.124,-0.2,0.30,0.00,-0.01,0.62,...,-0.138,-1.0,0.2,-0.7,0.156667,0.356667,0.016667,-0.095333,-0.333333,0.51


In [14]:
#dataset creation ensofeatures
features.to_csv(
    PROCESSED_DATA / "enso_features.csv",
    index=False
)
list(PROCESSED_DATA.iterdir())

[PosixPath('../data/processed/enso_features.csv'),
 PosixPath('../data/processed/master_enso_monthly.csv')]